In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# ==========================================================
# Configuration
# ==========================================================

SILVER_TABLE     = "finance_intelligence_hub.silver.stock_prices"
COMPANIES_TABLE  = "finance_intelligence_hub.silver.companies"
GOLD_VIEW        = "finance_intelligence_hub.gold.fact_stock_prices"


print("="*80)
print("Stock Prices Gold View")
print("="*80)



spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_VIEW} AS
SELECT
    ROW_NUMBER() OVER (ORDER BY sp.trade_date, sp.ticker) AS Index,
    sp.ticker,
    sp.trade_date,
    sp.open_price,
    sp.high_price,
    sp.low_price,
    sp.close_price,
    LAG(sp.close_price) OVER (
        PARTITION BY sp.ticker
        ORDER BY sp.trade_date
    ) AS previous_close_price,
    sp.adjusted_close_price,
    sp.volume
FROM {SILVER_TABLE} sp
LEFT SEMI JOIN {COMPANIES_TABLE} c
    ON sp.ticker = c.ticker
""")
print(f"Gold view {GOLD_VIEW} created/refreshed successfully.")

# ==========================================================
# Quick sanity check
# ==========================================================

row_count = spark.sql(f"SELECT COUNT(*) FROM {GOLD_VIEW}").first()[0]
print(f"Gold view row count: {row_count:,}")

spark.sql(f"DESCRIBE {GOLD_VIEW}").show(truncate=False)